## Predict 5′ cap composition from biological fingerprint data

This notebook predicts the most likely 5′ cap composition of an unknown biological sample using precomputed training libraries.

The user only needs to provide:

1. Path to the biological fingerprint CSV.
2. Sample name / experiment name.
3. Whether to use:
   - `exclude_zero_caps=True`
   - `exclude_zero_caps=False`
   - `include_insdel=True`
   - `include_insdel=False`

No training mixture generation is performed in this notebook.

### 0. Load libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rnacappredictor.predict_cap import predict_cap

### 1. Configuration
#### Load fingerprints.csv files and define the experiment name as the SAMPLE_BATCH_NAME

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================

SAMPLE_BATCH_NAME = "FM219"

FINGERPRINT_PATHS = [
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-1_test/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-11/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-138P/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U1-148P/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U6/fingerprints.csv",
    "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U4/fingerprints.csv",
]

# For biological blind samples, I would use False as the first-pass default
EXCLUDE_ZERO_CAPS = False

# Use True only if you want the INS/DEL fingerprint columns
INCLUDE_INSDEL = False

PRINT_TOP_K = 15

OUTPUT_DIR = "../results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### 2. Select the precomputed training library

In [ ]:
# ============================================================
# PRECOMPUTED TRAINING LIBRARIES
# ============================================================

MIXES_TRUE_PATH = "../models/df_train_mixes_exclude_zero_TRUE_step002.parquet"
MIXES_TRUE_PATH_INSDEL = "../models/df_train_mixes_exclude_zero_TRUE_INSDEL_step002.parquet"

MIXES_FALSE_PATH = "../models/df_train_mixes_exclude_zero_FALSE_step002.parquet"
MIXES_FALSE_PATH_INSDEL = "../models/df_train_mixes_exclude_zero_FALSE_INSDEL_step002.parquet"

if EXCLUDE_ZERO_CAPS and INCLUDE_INSDEL:
    MIXES_PATH = MIXES_TRUE_PATH_INSDEL
    MODE_DESCRIPTION = "exclude_zero_TRUE_INSDEL_step002"

elif EXCLUDE_ZERO_CAPS and not INCLUDE_INSDEL:
    MIXES_PATH = MIXES_TRUE_PATH
    MODE_DESCRIPTION = "exclude_zero_TRUE_step002"

elif not EXCLUDE_ZERO_CAPS and INCLUDE_INSDEL:
    MIXES_PATH = MIXES_FALSE_PATH_INSDEL
    MODE_DESCRIPTION = "exclude_zero_FALSE_INSDEL_step002"

else:
    MIXES_PATH = MIXES_FALSE_PATH
    MODE_DESCRIPTION = "exclude_zero_FALSE_step002"

print("Selected model/library:")
print(MODE_DESCRIPTION)
print(MIXES_PATH)

### 3. Load the saved training mixtures

In [ ]:
# ============================================================
# LOAD PRECOMPUTED TRAINING MIXTURES
# ============================================================

df_train_mixes = pd.read_parquet(MIXES_PATH)

print("Loaded training mixture library:")
print("Shape:", df_train_mixes.shape)
print("Columns:", df_train_mixes.columns.tolist())
print("RTs:", df_train_mixes["RT"].unique())

### 4. Load and concatenate multiple fingerprint files
This function is to be added to predict_cap.py file 

In [ ]:
# ============================================================
# LOAD BIOLOGICAL FINGERPRINT FILES
# ============================================================

def load_fingerprint_files(fingerprint_paths):
    dfs = []

    for path in fingerprint_paths:
        temp = pd.read_csv(path)
        temp["source_file"] = path
        dfs.append(temp)

    return pd.concat(dfs, ignore_index=True)


df_test = load_fingerprint_files(FINGERPRINT_PATHS)

print("Loaded biological fingerprints:")
print("Shape:", df_test.shape)
display(df_test.head())
print("Columns:", df_test.columns.tolist())

### 5. Barcode-to-RT mapping

In [ ]:
# ============================================================
# BARCODE TO RT MAPPING
# ============================================================

barcode_isoform_to_rt = {
    (1, "U1-1"): "INDURO",
    (6, "U1-1"): "ProtoScript",
    (11, "U1-1"): "Marathon",
    (16, "U1-1"): "GoScript",
    (21, "U1-1"): "EpiScript",

    (4, "U1-11"): "INDURO",
    (9, "U1-11"): "ProtoScript",
    (14, "U1-11"): "Marathon",
    (19, "U1-11"): "GoScript",
    (24, "U1-11"): "EpiScript",

    (2, "U1-138P"): "INDURO",
    (7, "U1-138P"): "ProtoScript",
    (12, "U1-138P"): "Marathon",
    (17, "U1-138P"): "GoScript",
    (22, "U1-138P"): "EpiScript",

    (3, "U1-148P"): "INDURO",
    (8, "U1-148P"): "ProtoScript",
    (13, "U1-148P"): "Marathon",
    (18, "U1-148P"): "GoScript",
    (23, "U1-148P"): "EpiScript",

    (5, "U6"): "INDURO",
    (10, "U6"): "ProtoScript",
    (15, "U6"): "Marathon",
    (20, "U6"): "GoScript",
    (1, "U6"): "EpiScript",

    (5, "U4"): "INDURO",
    (10, "U4"): "ProtoScript",
    (15, "U4"): "Marathon",
    (20, "U4"): "GoScript",
    (1, "U4"): "EpiScript",
}

### 6. Prepare the concatenated dataframe

In [ ]:
# ============================================================
# PREPARE TEST DATA
# ============================================================

df_test = df_test.copy()

# Convert barcode from "barcode01" / "barcode1" to integer if needed
if "barcode" in df_test.columns:
    df_test["barcode"] = df_test["barcode"].apply(
        lambda x: int(str(x).replace("barcode", ""))
    )

# Add RT using barcode + isoform
df_test["RT"] = df_test.apply(
    lambda row: barcode_isoform_to_rt[(row["barcode"], row["isoform"])],
    axis=1
)

# Unknown cap label required by create_features()
df_test["cap"] = "unknown"

# CRITICAL:
# Use a separate experiment name per isoform/RNA.
# Otherwise all RNAs will be treated as one sample.
df_test["experiment"] = SAMPLE_BATCH_NAME + "_" + df_test["isoform"].astype(str)

print("Prepared test data:")
display(df_test.head())
print("Shape:", df_test.shape)
print()
print("Experiments to predict:")
print(df_test["experiment"].value_counts())
print()
print("RTs by experiment:")
display(df_test.groupby("experiment")["RT"].unique())

### 7. Validate columns 

In [ ]:
# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

if INCLUDE_INSDEL:
    required_columns = [
        "RT",
        "A%_INSDEL",
        "C%_INSDEL",
        "G%_INSDEL",
        "T%_INSDEL",
        "INS%_INSDEL",
        "DEL%_INSDEL",
        "cap",
        "experiment",
    ]
else:
    required_columns = [
        "RT",
        "A%",
        "C%",
        "G%",
        "T%",
        "cap",
        "experiment",
    ]

missing_columns = [col for col in required_columns if col not in df_test.columns]

if missing_columns:
    raise ValueError(
        "Missing required columns in biological fingerprint dataframe: "
        + ", ".join(missing_columns)
    )

print("All required columns are present.")

### 8. Check for unmapped barcodes

In [ ]:
# ============================================================
# CHECK BARCODE/ISOFORM MAPPING
# ============================================================

observed_pairs = set(zip(df_test["barcode"], df_test["isoform"]))

missing_pairs = [
    pair for pair in observed_pairs
    if pair not in barcode_isoform_to_rt
]

if missing_pairs:
    raise ValueError(
        "These barcode/isoform pairs are missing from barcode_isoform_to_rt:\n"
        + "\n".join(str(pair) for pair in sorted(missing_pairs))
    )

df_test["RT"] = df_test.apply(
    lambda row: barcode_isoform_to_rt[(row["barcode"], row["isoform"])],
    axis=1
)

### 9. Check for RT compatibility

In [ ]:
# ============================================================
# CHECK RT COMPATIBILITY
# ============================================================

training_rts = set(df_train_mixes["RT"].unique())
test_rts = set(df_test["RT"].unique())

missing_in_test = training_rts - test_rts
unknown_in_test = test_rts - training_rts

print("RTs in training:", sorted(training_rts))
print("RTs in test:", sorted(test_rts))

if missing_in_test:
    print()
    print("Warning: these RTs are present in training but missing from the test:")
    print(sorted(missing_in_test))

if unknown_in_test:
    print()
    print("Warning: these RTs are present in test but missing from training:")
    print(sorted(unknown_in_test))

### 10. Run predictions for all isoforms at once

In [ ]:
# ============================================================
# RUN PREDICTION
# ============================================================

results = predict_cap(
    df_train_mixes,
    df_test,
    show_true_cap=False,
    include_insdel=INCLUDE_INSDEL,
    print_top_k=PRINT_TOP_K,
    save_model=False
)

display(results)